# Protein Pocket Prediction Pipeline

1. **InterPro domain processing**: Extract domain structures and cluster sequences
3. **Pocket prediction**: Run Fpocket and P2Rank
4. **Database creation**: Characterize pockets and build final database

## Requirements

- Python packages: biopython, pandas, numpy, tqdm, requests
- External tools: CD-HIT, Fpocket, P2Rank
- Input: AlphaFold PDB files in `raw/pdb/` directory

## Setup and Imports

In [ ]:
from collections import defaultdict
import gzip
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import pandas as pd
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from pathlib import Path
from tqdm import tqdm
import shutil

from pathlib import Path
import shutil
from collections import defaultdict
from Bio import SeqIO
from tqdm import tqdm
import subprocess
import pandas as pd
import requests
import time

import json
import numpy as np
import re
from pathlib import Path
import subprocess
from tqdm import tqdm
import multiprocessing
import shutil

## Configuration

In [ ]:
# Directory structure
ALPHAFOLD_PDB_DIR = Path('raw/pdb')
PAE_DIR = Path('raw/pae')

# InterPro files
INTERPRO_DIR = Path('../databases/20250205_interpro')
PROTEIN2IPR_PATH = INTERPRO_DIR / "protein2ipr.dat.gz"
ENTRYLIST_PATH = INTERPRO_DIR / "entry.list"

# CDHIT
CLUSTERING_CUTOFF = 0.90 # clustering identity threshold
CDHIT_PATH = '/home/isglobal/SOFTWARE/cd-hit-v4.8.1-2019-0228/cd-hit' # path to binary

# Pocket prediction
FPOCKET_PATH = '/home/isglobal/SOFTWARE/fpocket/bin/fpocket'
P2RANK_PATH = '/home/isglobal/SOFTWARE/p2rank_2.4/prank'
FPOCKET_RESULTS_DIR = Path('processed/fpocket')
P2RANK_RESULTS_DIR = Path('processed/p2rank')

# Gene annotation (from TriTrypDB)
GENE_ANNOTATION_CSV = 'raw/gene_annotation.csv'

for dir_path in [PAE_DIR, FPOCKET_RESULTS_DIR, P2RANK_RESULTS_DIR]: 
    dir_path.mkdir(parents=True, exist_ok=True)

---
# Step 1: Domain annotations from InterPro

Get domain annotations for our AlphaFold proteins.

Needs the protein2ipr.dat.gz and entry.list files from InterPro already downloaded.

In [ ]:
# Read InterPro entries
entry_df = pd.read_csv(ENTRYLIST_PATH, sep="\t")

# Iterate through AF files
proteins_of_interest = []
pdb_files = ALPHAFOLD_PDB_DIR.glob('./*.pdb')
for pdb_file in pdb_files:
    # this pressuposes that the file names have not been changed
    # current format: AF-Q2VLK8-F1-model_v6.pdb
    protein_id = pdb_file.stem.split('-')[1]  
    proteins_of_interest.append(protein_id) 
proteins_of_interest = set(proteins_of_interest)

domains_dict = defaultdict(list)  # more than one domain per protein
with gzip.open(PROTEIN2IPR_PATH, 'rt') as f:  # it takes some time, it's a big file
    for chunk in tqdm(pd.read_csv(f,
                                  sep='\t',
                                  header=None,
                                  names=["protein_id","interpro_id","interpro_name","db_id","start","end"],
                                  chunksize=25_000_000),
                      desc="Processing chunks"
                      ):
        # filter by proteins of interest
        chunk = chunk[chunk['protein_id'].isin(proteins_of_interest)]
        if chunk.empty:
            continue
        
        # append rows to dictionary
        for _, row in chunk.iterrows():
            domains_dict[row['protein_id']].append(
                (row['interpro_id'],
                 row['interpro_name'],
                 row['db_id'],
                 row['start'],
                 row['end']
                 )
            )
            
# Convert to df
rows = []
for prot, domains in domains_dict.items():
    for interpro, name, db_id, start, end in domains:
        rows.append([prot, interpro, name, db_id, start, end])
df = pd.DataFrame(rows, columns=["protein_id", "interpro_id", "interpro_name", 'db_id', "start", "end"])

# Add entries data
df = (df
      .merge(entry_df[["ENTRY_AC", "ENTRY_TYPE"]], left_on="interpro_id", right_on="ENTRY_AC")
      .rename(columns={'ENTRY_TYPE': 'entry_type'})
      .drop(columns=['ENTRY_AC'])
      )
allowed_types = ["Domain", "Family", "Homologous_superfamily"]
df = df[df["entry_type"].isin(allowed_types)]

# In cases of multiple instances of domains in a protein, keep the min as start and max as end positions
interpro_df = (df
               .groupby(['protein_id', 'interpro_id', 'interpro_name', 'entry_type'], as_index=False)
               .agg({'start': 'min', 'end': 'max'})
               )
interpro_df.to_csv('processed/interpro_data.csv', index=False)

---
# Stage 2: Sequence clustering

Extract sequences from the AF PDB files, and cluster sequences to reduce redundancy (we want the fewest number of structures possible)

### Full sequences from PDB files

In [ ]:
interpro_df = pd.read_csv('processed/interpro_data.csv')

protein_list = set(interpro_df['protein_id'])  # only keep those proteins with interpro annotation
full_seqs = {}
pdb_parser = PDBParser(QUIET=True)
for protein_id in tqdm(protein_list):
    pdb_file = ALPHAFOLD_PDB_DIR / f"AF-{protein_id}-F1-model_v6.pdb"  # careful if using other AF version
    structure = pdb_parser.get_structure(protein_id, str(pdb_file))
    seq = [seq1(residue.get_resname()) for residue in structure.get_residues()]
    seq = ''.join(seq)
    full_seqs[protein_id] = seq

full_fasta_path = Path('processed/protein_sequences.fasta')
with open(full_fasta_path, "w+") as f:
    for protein_id, sequence in full_seqs.items():
        seq_record = SeqRecord(Seq(sequence), id=protein_id, description="")
        SeqIO.write(seq_record, f, "fasta")

### Sequence clustering with CD-HIT

In [ ]:
def run_cd_hit(input_fasta,
               output,
               cutoff = 0.9,
               cdhit_path = "cd-hit"
               ):
    """
    Run CD-HIT on the input FASTA file to cluster similar sequences.
    """
    print("Clustering sequences with CD-HIT...")
    cmd = [cdhit_path,
           "-i", str(input_fasta),
           "-o", str(output),
           "-c", str(cutoff),  # sequence identity threshold
           "-g", "1",          # more accurate clustering
           "-d", "0",          # full-length headers in .clstr output
           "-sc", "1"          # sort clusters by size
           ]
    
    # Run the CD-HIT command
    subprocess.run(cmd, check=True)

proteins_fasta = Path("processed/protein_sequences.fasta")
cutoff_str = str(int(CLUSTERING_CUTOFF*100)).replace(".", "")  # e.g. "90"
clstr_output = f'processed/clstr_{cutoff_str}'

run_cd_hit(input_fasta=proteins_fasta,
           output=clstr_output,
           cutoff=CLUSTERING_CUTOFF,
           cdhit_path=CDHIT_PATH
           )

records = SeqIO.parse(clstr_output, 'fasta')
ref_proteins = [record.id for record in records]  # get the ref seq for each cluster
with open(f'processed/clstr_{cutoff_str}_ref_ids.txt', 'w+') as f:
       f.write("\n".join(ref_proteins))
print(f"Obtained {len(ref_proteins)} reference proteins")

---
# Step 3: Pocket prediction

Run Fpocket for pocket detection and P2Rank for re-scoring.

### Pocket detection with fpocket

In [ ]:
def run_fpocket(args):
    pdb_file, fpocket_results_dir, fpocket_exe = args
    """Run fpocket on a single PDB file and move results to output directory."""
    results_folder = pdb_file.parent / f"{pdb_file.stem}_out"

    # Clean up any existing output for this file
    if results_folder.exists():
        shutil.rmtree(results_folder)

    # Run fpocket
    cmd = [fpocket_exe, '-f', str(pdb_file.resolve())]
    subprocess.run(cmd,
                   check=True,
                   stdout=subprocess.DEVNULL,
                   stderr=subprocess.DEVNULL,
                   )

    # Move the results folder to fpocket dir
    if results_folder.exists():
        destination = fpocket_results_dir / results_folder.name
        if destination.exists():
            shutil.rmtree(destination)
        shutil.move(str(results_folder), str(destination))


cutoff_str = str(int(CLUSTERING_CUTOFF*100)).replace(".", "")  # e.g. "90"
with open(f'processed/clstr_{cutoff_str}_ref_ids.txt') as f:
    protein_list = [l.strip() for l in f]

pdb_files = {x.stem.split('-')[1]: x
             for x in ALPHAFOLD_PDB_DIR.glob('./*.pdb')
             if x.stem.split('-')[1] in protein_list
             }


tasks = [(pdb_path, FPOCKET_RESULTS_DIR, FPOCKET_PATH)
         for pdb_id, pdb_path in pdb_files.items()
         ]

with multiprocessing.Pool(processes=multiprocessing.cpu_count()) as pool:
    list(tqdm(pool.imap_unordered(run_fpocket, tasks), total=len(tasks)))

### P2Rank re-scoring

In [32]:
# Build the fpocket_list.ds file for p2rank
fpocket_entries = []
for folder in FPOCKET_RESULTS_DIR.iterdir(): # Scan all *_out folders
    if folder.is_dir() and folder.name.endswith('_out'):
        prediction_pdb = folder / f"{folder.name}.pdb"
        if prediction_pdb.exists():
            ori_pdb = ALPHAFOLD_PDB_DIR / f"{folder.name.replace('_out', '')}.pdb"
            fpocket_entries.append(f"{prediction_pdb}  {ori_pdb}")

# Write db file
fpocket_list_file = Path('fpocket_list.ds') # needs to be in current dir...
with open(fpocket_list_file, 'w+') as f:
    f.write("PARAM.PREDICTION_METHOD=fpocket\n\n")
    f.write("HEADER: prediction protein\n\n")
    for entry in fpocket_entries:
        f.write(f"{entry}\n")
print(f"fpocket_list file created with {len(fpocket_entries)} entries.")

# Empty results dir
for f in P2RANK_RESULTS_DIR.glob('./*'):
    if f.is_file():
        f.unlink()
    else:
        shutil.rmtree(f)

# Run P2Rank
print("Running P2Rank re-scoring...")
cmd = [P2RANK_PATH,
       "rescore",
       fpocket_list_file,
       "-c", "alphafold",
       "-o", str(P2RANK_RESULTS_DIR),
       ]

subprocess.run(cmd, check=True)
print("P2Rank re-scoring complete!")

processing [AF-Q4CZQ2-F1-model_v6_out.pdb] (2310/8721)
processing [AF-Q4DW53-F1-model_v6_out.pdb] (2311/8721)
processing [AF-Q4DUD8-F1-model_v6_out.pdb] (2312/8721)
processing [AF-Q4DBB5-F1-model_v6_out.pdb] (2313/8721)
processing [AF-Q4CLG2-F1-model_v6_out.pdb] (2314/8721)
processing [AF-Q4CMZ8-F1-model_v6_out.pdb] (2315/8721)
processing [AF-Q4D7V3-F1-model_v6_out.pdb] (2316/8721)
processing [AF-Q4E096-F1-model_v6_out.pdb] (2317/8721)
processing [AF-Q4DUQ5-F1-model_v6_out.pdb] (2318/8721)
processing [AF-Q4DPG2-F1-model_v6_out.pdb] (2319/8721)
processing [AF-Q4DTU3-F1-model_v6_out.pdb] (2320/8721)
processing [AF-Q4E410-F1-model_v6_out.pdb] (2321/8721)
processing [AF-Q4DEJ5-F1-model_v6_out.pdb] (2322/8721)
processing [AF-Q4D7U7-F1-model_v6_out.pdb] (2323/8721)
processing [AF-Q4DSI7-F1-model_v6_out.pdb] (2324/8721)
processing [AF-Q4DJ81-F1-model_v6_out.pdb] (2325/8721)
processing [AF-Q4D1F8-F1-model_v6_out.pdb] (2326/8721)
processing [AF-Q4D4Q4-F1-model_v6_out.pdb] (2327/8721)
processing

---
# Step 4: Pocket characterization and database creation

Parse pocket predictions and compute features for the final database.

In [33]:
def parse_pqr_file(file_path):
    pocket_data = {}
    atoms = []

    # Regular expressions for header and atom lines
    header_regex = re.compile(r'HEADER\s+(\d+)\s*-\s*(.*?):\s*(.+)')

    with open(file_path, 'r') as file:
        for line in file:
            
            if line.startswith('HEADER'):
                # Match header information
                header_match = header_regex.match(line)
                if header_match:
                    key = header_match.group(2).strip()
                    value = float(header_match.group(3).strip())
                    pocket_data[key] = value
            
            elif line.startswith('ATOM'):
                atom_number = int(line[6:11].strip())
                atom_type = line[12:16].strip()
                residue_name = line[17:20].strip()
                residue_id = int(line[22:26].strip())
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())
                charge = float(line[54:60].strip())
                radius = float(line[60:66].strip())                
                    
                atoms.append({
                    'atom_number': atom_number,
                    'atom_type': atom_type,
                    'residue_name': residue_name,
                    'residue_id': residue_id,
                    'coord': (x, y, z),
                    'charge': charge,
                    'radius': radius
                })
    if not atoms:
        print(f"No voronoi vertices found in {file_path.stem}!")
        return pocket_data, None
    return pocket_data, atoms


def calculate_centroid(atoms_coords):
    total_x = total_y = total_z = 0.0
    atom_count = len(atoms_coords)

    if atom_count == 0:
        return None  # Return None if no atoms are provided

    for atom_coord in atoms_coords:
        x, y, z = atom_coord
        total_x += x
        total_y += y
        total_z += z

    # Calculate the average for each coordinate
    center_x = np.round(total_x / atom_count, 3)
    center_y = np.round(total_y / atom_count, 3)
    center_z = np.round(total_z / atom_count, 3)

    return center_x, center_y, center_z


def get_plddts(pdb_structure):
    plddt_values = {}

    for model in pdb_structure:
        for chain in model:
            for residue in chain:
                residue_id = residue.get_id()[1]
                for atom in residue:
                    if atom.get_name() == 'CA':  # Only consider C-alpha atoms for pLDDT
                        plddt_values[residue_id] = atom.bfactor
                        break  # No need to check other atoms in the residue
    return plddt_values


def calculate_pae(residue_ids, pae_matrix):
    residue_ids = set(residue_ids)  
    pae_means = {}
    for base_residue, pae_values in enumerate(pae_matrix, 1):
        
        if base_residue not in residue_ids:
            continue
    
        aligned_pae = [pae_value
                       for ix, pae_value in enumerate(pae_values, 1)
                       if ix in residue_ids and ix != base_residue
                       ]
        residue_pae_mean = np.array(aligned_pae).mean(dtype=np.float64)
        pae_means[base_residue] = residue_pae_mean

    return pae_means


# Function to process each protein file
def process_protein(args):
    pdb_file, domains_dict, pae_file, fpocket_pred_dir, p2rank_rescore_file = args
    assert pdb_file.is_file()
    assert pae_file.is_file()
    assert p2rank_rescore_file.is_file()
    
    protein_id = pdb_file.stem.split('-')[1]
    results = []

    # Get the domains for this protein
    protein_domains = domains_dict.get(protein_id, [])

    # Get pLDDT values from the PDB file
    try:
        full_structure = pdb_parser.get_structure('whole', pdb_file)
        plddts = get_plddts(full_structure)
    except Exception as e:
        print(f"Error reading PDB file {protein_id}: {e}")
        return results

    # Parse PAE matrix
    try:
        with open(pae_file) as f:
            pae_data = json.load(f)[0]
    except (FileNotFoundError, json.JSONDecodeError):
        print(f"Error loading PAE file for {protein_id}, skipping...")
        return results
    pae_matrix = np.array(pae_data["predicted_aligned_error"])

    # Get fpocket predictions
    fpocket_pred_dir = fpocket_pred_dir / 'pockets'
    if not fpocket_pred_dir.is_dir():
        print(f"Error: Pocket files not found for {protein_id}, skipping...")
        return results

    # Get P2Rank re-scoring
    try:
        p2rank_df = pd.read_csv(p2rank_rescore_file, index_col=0)
        p2rank_dict = p2rank_df.to_dict('index')
    except (FileNotFoundError, pd.errors.EmptyDataError):
        print(f"Error reading P2Rank file for {protein_id}, skipping...")
        return results

    # Iterate through all pockets
    for pqr_file in fpocket_pred_dir.glob('*.pqr'):
        pocket_num = pqr_file.stem.split('_')[0]
        pocket_res_file = fpocket_pred_dir / f'{pocket_num}_atm.pdb'

        # Get pocket information
        pocket_info, points = parse_pqr_file(pqr_file)
        if points is None:
            print(f"Error reading Fpocket file {pqr_file} skipping...")
            continue  # Skip if no points found

        points_coords = [p['coord'] for p in points]
        p_x, p_y, p_z = calculate_centroid(points_coords)
        
        # Get nearby residues
        try:
            pocket_resis = [r for r in pdb_parser.get_structure('pocket_resis', pocket_res_file).get_residues()]
            resis_ids = [r.id[1] for r in pocket_resis]
        except Exception as e:
            print(f"Error reading pocket residues for {protein_id}, skipping pocket {pocket_num}: {e}")
            continue

        # Calculate domain coverage
        dom_dict = {}
        for dom in protein_domains:
            _, dname, dstart, dend = dom.split('_')
            res_in_range = [res for res in resis_ids if int(dstart) <= res <= int(dend)]
            dom_dict[dname] = np.round(len(res_in_range) / len(resis_ids), 2)
        max_prop = max(dom_dict.values(), default=0)

        # Calculate pLDDT and PAE
        pocket_plddts = [plddts.get(resid, 0) for resid in resis_ids]  # Default to 0 if residue not found
        mean_plddt = np.mean(pocket_plddts)
        median_plddt = np.median(pocket_plddts)

        pocket_paes = calculate_pae(resis_ids, pae_matrix)
        mean_pae = np.mean(list(pocket_paes.values()))
        median_pae = np.median(list(pocket_paes.values()))
    
        # Get P2Rank scores
        p2rank_key = f'pocket.{pocket_num.split("pocket")[1]}'
        if p2rank_key not in p2rank_dict:
            print(f"Error: P2Rank key not found for pocket {pocket_num} in {protein_id}, skipping...")
            continue

        p2rank_rescore = p2rank_dict[p2rank_key].get('score', 0)
        p2rank_rank = p2rank_dict[p2rank_key].get('rank', 0)

        entry = {
            'protein_id': protein_id,
            'pocket_id': pocket_num,
            'centroid_x': p_x,
            'centroid_y': p_y,
            'centroid_z': p_z,
            'fpocket_rank': int(pocket_num.split("pocket")[1]),
            'p2rank_rank': p2rank_rank,
            'mean_plddt': np.round(mean_plddt, 2),
            'median_plddt': np.round(median_plddt, 2),
            'mean_pae': np.round(mean_pae, 2),
            'median_pae': np.round(median_pae, 2),
            'max_domain_prop': max_prop,
            'p2rank_score': p2rank_rescore,
            'fpocket_score': float(pocket_info.get('Pocket Score', 0)),
            'fpocket_drug': float(pocket_info.get('Drug Score', 0)),
        }

        results.append(entry)

    return results

### Download PAE matrices for clustered proteins

In [36]:
# Get list of PDB files
cutoff_str = str(int(CLUSTERING_CUTOFF*100)).replace(".", "")  # e.g. "90"
with open(f'processed/clstr_{cutoff_str}_ref_ids.txt') as f:
    protein_list = [l.strip() for l in f]
    
pdb_files = {x.stem.split('-')[1]: x
             for x in ALPHAFOLD_PDB_DIR.glob('./*.pdb')
             if x.stem.split('-')[1] in protein_list
             }

# We also searched for Interpro annotations of proteins
domains_df = pd.read_csv('processed/interpro_data.csv', index_col=0)
domains_dict = defaultdict(list)
for protein_id, df in domains_df.groupby('protein_id'):
    for _, row in df.iterrows():
        domain_str = '_'.join([protein_id, row['interpro_id'], str(row['start']), str(row['end'])])
        domains_dict[protein_id].append(domain_str)

# We also need the PAE of each file
print(f"Downloading PAE matrices for {len(pdb_files)} proteins...")
for pdb_id, pdb_file in tqdm(pdb_files.items(), desc="Downloading PAE files"):
    pdb_prefix = pdb_file.stem.rsplit("-", 1)[0]
    error_fname = f"{pdb_prefix}-predicted_aligned_error_v6.json"
    error_url = f"https://alphafold.ebi.ac.uk/files/{error_fname}"
    output_file = PAE_DIR / error_fname

    if output_file.is_file():
        continue
    
    try:
        response = requests.get(error_url)
        error_json = response.text
    except Exception as e:
        print(f"Error downloading {pdb_id}, retrying...")
        time.sleep(10)
        response = requests.get(error_url)
        error_json = response.text
    
    with open(output_file, 'w+') as f:
        f.write(error_json)

### Process the pockets

In [35]:
# Use multiprocessing to process PDB files in parallel
tasks = [(pdb_path,
          domains_dict,
          PAE_DIR / f"AF-{pdb_id}-F1-predicted_aligned_error_v6.json",
          FPOCKET_RESULTS_DIR / f'{pdb_path.stem}_out',
          P2RANK_RESULTS_DIR / f'{pdb_path.stem}_out.pdb_rescored.csv',
          )
         for pdb_id, pdb_path in pdb_files.items()
         ]

with multiprocessing.Pool(10) as pool:
    all_pockets = list(tqdm(pool.imap_unordered(process_protein, tasks), total=len(pdb_files)))
flat_pockets = [item for sublist in all_pockets for item in sublist]

100%|██████████| 7087/7087 [02:53<00:00, 40.74it/s]


In [37]:
# Create dataframe and calculate z-scores
pockets_df = pd.DataFrame(flat_pockets)
if not pockets_df.empty:
    mean_score = pockets_df['p2rank_score'].mean()
    std_score = pockets_df['p2rank_score'].std()
    pockets_df['z_score'] = (pockets_df['p2rank_score'] - mean_score) / std_score
else:
    print("No valid pockets found, nothing to save.")
    
genes_df = pd.read_csv(GENE_ANNOTATION_CSV)
organism_priority = {'Trypanosoma cruzi CL Brener Esmeraldo-like': 0,
                     'Trypanosoma cruzi CL Brener Non-Esmeraldo-like': 1,
                     'Trypanosoma cruzi strain CL Brener': 2
                     }
genes_df['_priority'] = genes_df['Organism'].map(organism_priority)
genes_df = (genes_df
            .sort_values('_priority')
            .drop_duplicates(subset=['ext_id'], keep='first')
            .drop(columns='_priority')
            .reset_index(drop=True)
            )

pockets_df = pd.merge(pockets_df, genes_df, left_on='protein_id', right_on='ext_id')

pockets_df.to_csv("db/proteins/all_pockets.csv")
pockets_df.to_pickle("db/proteins/all_pockets.pkl")
pockets_df

,protein_id,pocket_id,centroid_x,centroid_y,centroid_z,fpocket_rank,p2rank_rank,mean_plddt,median_plddt,mean_pae,...,Superfamily Description,TigrFam Description,Computed GO Components,Computed GO Functions,Computed GO Processes,Curated GO Components,Curated GO Functions,Curated GO Processes,EC numbers,EC numbers from OrthoMCL
0,Q4DZM9,pocket4,-1.847,17.659,1.146,4,8,95.42,95.56,2.14,...,P-loop containing nucleoside triphosphate hydr...,Small GTP-binding protein domain,NaN,GTP binding;GTPase activity,protein transport,spliceosomal complex,NaN,NaN,NaN,NaN
1,Q4DZM9,pocket6,5.717,14.121,7.491,6,6,96.33,96.40,1.43,...,P-loop containing nucleoside triphosphate hydr...,Small GTP-binding protein domain,NaN,GTP binding;GTPase activity,protein transport,spliceosomal complex,NaN,NaN,NaN,NaN
2,Q4DZM9,pocket2,1.637,10.299,-9.457,2,1,95.60,97.00,1.92,...,P-loop containing nucleoside triphosphate hydr...,Small GTP-binding protein domain,NaN,GTP binding;GTPase activity,protein transport,spliceosomal complex,NaN,NaN,NaN,NaN
3,Q4DZM9,pocket11,-15.478,6.220,7.968,11,10,96.60,97.00,1.24,...,P-loop containing nucleoside triphosphate hydr...,Small GTP-binding protein domain,NaN,GTP binding;GTPase activity,protein transport,spliceosomal complex,NaN,NaN,NaN,NaN
4,Q4DZM9,pocket3,14.629,-7.645,6.516,3,12,97.72,97.88,1.10,...,P-loop containing nucleoside triphosphate hydr...,Small GTP-binding protein domain,NaN,GTP binding;GTPase activity,protein transport,spliceosomal complex,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311657,Q4D8L4,pocket67,-18.495,-1.395,20.362,67,106,94.08,94.94,1.53,...,Ribosomal protein S5 domain 2-type fold;Histid...,NaN,NaN,ATP binding;DNA binding;DNA topoisomerase type...,DNA topological change,nucleus,NaN,NaN,5.99.1.3 (Transferred entry: 5.6.2.2),1.1.1.37 (Malate dehydrogenase);1.3.1.74 (2-al...
311658,Q4D8L4,pocket50,44.935,6.491,-4.513,50,24,71.41,79.06,7.76,...,Ribosomal protein S5 domain 2-type fold;Histid...,NaN,NaN,ATP binding;DNA binding;DNA topoisomerase type...,DNA topological change,nucleus,NaN,NaN,5.99.1.3 (Transferred entry: 5.6.2.2),1.1.1.37 (Malate dehydrogenase);1.3.1.74 (2-al...
311659,Q4D8L4,pocket14,-27.019,21.109,51.345,14,58,91.67,91.62,2.10,...,Ribosomal protein S5 domain 2-type fold;Histid...,NaN,NaN,ATP binding;DNA binding;DNA topoisomerase type...,DNA topological change,nucleus,NaN,NaN,5.99.1.3 (Transferred entry: 5.6.2.2),1.1.1.37 (Malate dehydrogenase);1.3.1.74 (2-al...
311660,Q4D8L4,pocket79,50.135,-7.722,-54.061,79,51,83.66,83.16,3.57,...,Ribosomal protein S5 domain 2-type fold;Histid...,NaN,NaN,ATP binding;DNA binding;DNA topoisomerase type...,DNA topological change,nucleus,NaN,NaN,5.99.1.3 (Transferred entry: 5.6.2.2),1.1.1.37 (Malate dehydrogenase);1.3.1.74 (2-al...


In [38]:
pdb_db_dir = Path('db/proteins/pdb')
pdb_db_dir.mkdir(exist_ok=True, parents=True)

for protein_id in set(pockets_df['protein_id']):
    dest_file = pdb_db_dir / f'AF-{protein_id}-F1-model_v6.pdb'
    if not dest_file.is_file():
        ori_file = ALPHAFOLD_PDB_DIR / f'AF-{protein_id}-F1-model_v6.pdb'
        assert ori_file.is_file()
        shutil.copy(ori_file, dest_file)